# Book Recommendation System

An offline, genre-based example using Python’s standard library. The small sample below is hand-curated metadata, not a bestseller dataset or a source of book prices. No machine-learning accuracy is claimed. Run every cell in order; no upload is needed for the demo.


In [ ]:
"""Small, offline genre-based book recommender; no trained model required."""
import argparse
import csv
from pathlib import Path

REQUIRED_COLUMNS = {"Book", "Author(s)", "Genre"}


def load_books(source):
    """Read a CSV path or text stream and validate its required columns."""
    if hasattr(source, "read"):
        reader = csv.DictReader(source)
        missing = REQUIRED_COLUMNS - set(reader.fieldnames or [])
        if missing:
            raise ValueError("Missing CSV columns: " + ", ".join(sorted(missing)))
        return [{key: (value or "").strip() for key, value in row.items()
                 if key is not None} for row in reader]
    with Path(source).open(encoding="utf-8-sig", newline="") as handle:
        return load_books(handle)


def recommend_books(books, genre, num_recommendations=5, *, max_price=None):
    """Return matching rows in source order. Unknown genres return an empty list.

    Optional price filtering requires a numeric Price column in the source currency.
    Missing/invalid prices are excluded, never assumed to mean free.
    """
    if not isinstance(genre, str) or not genre.strip():
        raise ValueError("Enter a non-empty genre.")
    if (isinstance(num_recommendations, bool)
            or not isinstance(num_recommendations, int) or num_recommendations < 1):
        raise ValueError("Number of recommendations must be a positive integer.")
    if max_price is not None:
        import math
        if not math.isfinite(max_price) or max_price < 0:
            raise ValueError("Maximum price must be a finite non-negative number.")
        if not books or not all("Price" in book for book in books):
            raise ValueError("Price filtering requires a Price column in the CSV.")
    matches = []
    for book in books:
        if (book.get("Genre") or "").strip().casefold() != genre.strip().casefold():
            continue
        if not book.get("Book"):
            continue
        if max_price is not None:
            try:
                price = float(book["Price"])
            except (ValueError, TypeError):
                continue
            if not 0 <= price <= max_price:
                continue
        matches.append(book)
        if len(matches) >= num_recommendations:
            break
    return matches



In [ ]:
import io
SAMPLE_CSV = "Book,Author(s),Genre\nThe Hobbit,J. R. R. Tolkien,Fantasy\nAlice's Adventures in Wonderland,Lewis Carroll,Fantasy\nPride and Prejudice,Jane Austen,Romance\nThe War of the Worlds,H. G. Wells,Science Fiction\nThe Hound of the Baskervilles,Arthur Conan Doyle,Mystery\n"
books = load_books(io.StringIO(SAMPLE_CSV))


## Your own CSV
Optionally replace `books` with `load_books("path/to/books.csv")`. Required columns: `Book`, `Author(s)`, `Genre`. Unknown genres return no results; blank genres are not guessed from language. Price filtering needs a numeric `Price` column in one consistent currency.


In [ ]:
genre = "Fantasy"
recommendations = recommend_books(books, genre, num_recommendations=5)
for book in recommendations:
    print(f"{book['Book']} — {book['Author(s)']}")
if not recommendations:
    print("No matching books. Try another genre.")


In [ ]:
assert recommend_books(books, "unknown") == []
assert len(recommend_books(books, " fantasy ", 1)) == 1
print("Basic checks passed.")
